# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# === Rebuild ML-08 pipeline in this notebook ===
from google.colab import userdata
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

def load_month_agg(month_str):
    files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
    month_files = [f for f in files if f"fact_content_daily_performance/month={month_str}" in f]
    local_paths = [
        hf_hub_download(repo_id="FlyRank/internship-warehouse", filename=f, repo_type="dataset")
        for f in month_files
    ]
    raw = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
    gsc = raw[raw['gsc_data_available'] == True].copy()
    agg = gsc.groupby(['client_hash_id', 'content_hash_id']).agg(
        days_with_data=('report_date', 'nunique'),
        total_impressions=('gsc_impressions', 'sum'),
        total_clicks=('gsc_clicks', 'sum'),
        avg_position=('gsc_avg_position', 'mean')
    ).reset_index()
    agg = agg[agg['total_impressions'] > 0].copy()
    agg['ctr'] = agg['total_clicks'] / agg['total_impressions']
    return agg

march = load_month_agg("2026-03")
april = load_month_agg("2026-04")

pos_bins = [0, 3, 10, 20, 50, 1000]
pos_labels = ['1-3', '4-10', '11-20', '21-50', '51+']

def add_ctr_gap(agg):
    agg = agg.copy()
    agg['position_bucket'] = pd.cut(agg['avg_position'], bins=pos_bins, labels=pos_labels)
    bucket_totals = agg.groupby('position_bucket', observed=True).agg(
        clicks_sum=('total_clicks', 'sum'), impressions_sum=('total_impressions', 'sum')
    )
    bucket_totals['expected_ctr'] = bucket_totals['clicks_sum'] / bucket_totals['impressions_sum']
    agg['expected_ctr'] = agg['position_bucket'].map(bucket_totals['expected_ctr']).astype(float)
    agg['ctr'] = agg['ctr'].astype(float)
    agg['ctr_gap'] = agg['expected_ctr'] - agg['ctr']
    agg['ctr_ratio'] = agg['ctr'] / agg['expected_ctr'].replace(0, np.nan)
    return agg

march = add_ctr_gap(march)
april = add_ctr_gap(april)
april['label_still_underperforming'] = (
    (april['total_impressions'] >= 1000) & (april['ctr_ratio'] < 0.5)
).astype(int)

def score_row(row):
    if row['total_impressions'] >= 1000 and pd.notnull(row['ctr_ratio']) and row['ctr_ratio'] < 0.5:
        return row['ctr_gap'] * row['total_impressions']
    return 0

march['baseline_score'] = march.apply(score_row, axis=1)

merged = march.merge(
    april[['client_hash_id', 'content_hash_id', 'label_still_underperforming']],
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

feature_cols = ['total_impressions', 'total_clicks', 'avg_position', 'ctr',
                 'ctr_gap', 'ctr_ratio', 'days_with_data']
merged_model = merged.dropna(subset=feature_cols + ['label_still_underperforming']).copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(merged_model, groups=merged_model['client_hash_id']))
train_df = merged_model.iloc[train_idx]
test_df = merged_model.iloc[test_idx]

X_train, y_train = train_df[feature_cols], train_df['label_still_underperforming']
X_test, y_test = test_df[feature_cols], test_df['label_still_underperforming']

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
test_df = test_df.copy()
test_df['model_score'] = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = []
for k in [20, 50, 100]:
    p_baseline = precision_at_k(test_df['baseline_score'], test_df['label_still_underperforming'], k)
    p_model = precision_at_k(test_df['model_score'], test_df['label_still_underperforming'], k)
    results.append({'k': k, 'baseline_precision': p_baseline, 'model_precision': p_model, 'base_rate': y_test.mean()})

comparison_table = pd.DataFrame(results)

print("Rebuild complete.")
print(f"merged_model: {len(merged_model)} rows")
print(f"train_df: {len(train_df)} rows | test_df: {len(test_df)} rows")
print(f"Test AUC: {roc_auc_score(y_test, test_df['model_score']):.3f}")
print(comparison_table)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rebuild complete.
merged_model: 157790 rows
train_df: 136739 rows | test_df: 21051 rows
Test AUC: 0.832
     k  baseline_precision  model_precision  base_rate
0   20                0.95             0.95    0.10218
1   50                0.98             0.80    0.10218
2  100                0.92             0.78    0.10218


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #3 — Click Capture by Position Tier (weighted CTR by position, 88% drop top-3 to deep).**

*Where does the label/metric come from?* Weighted CTR is computed by pooling total clicks over total impressions within each position tier, across all 341,701 pages and 57 brands — not by averaging each page's individual CTR. This is the right way to avoid the "average of ratios" trap the paper itself warns against elsewhere.

*Does the validation design carry the claim?* My question: since this is a pooled, portfolio-wide ratio rather than a per-brand comparison, how sensitive is the 88% drop to traffic concentration? If a small number of high-volume brands dominate the top-3 tier's impressions, the pooled ratio could mainly reflect those brands' behavior rather than a pattern shared broadly across all 57. It would strengthen the finding to show the same comparison computed as a median (or distribution) of per-brand weighted CTRs — if the drop holds up brand-by-brand and not just in the pooled number, that rules out one or two large accounts driving the whole result.

---

**Finding #10 — ML Appendix, Growth Prediction (logistic regression, 71% holdout accuracy).**

*Where does the label come from?* The growth/decline label is defined from 30-day-vs-previous-30-day impression trend — an observed outcome, not a rule-derived proxy, which is good practice per the framing skill.

*Does the validation design carry the claim?* The Methodology section states an 80/20 split for this model but doesn't specify whether it's a random row split or grouped by brand. With 61.8K content pieces across 57 brands, content from the same brand likely shares hidden structure — template, niche, typical position range — the same concern the leakage-hunting skill flags for my own ML-08 work. If the split was random rather than grouped by brand, part of the reported 71% holdout accuracy could reflect the model partially recognizing which brand a page belongs to, rather than a growth pattern that would generalize to a brand it's never seen. Naming the split type explicitly (and reporting the grouped-split number alongside the random one, if there's a gap) would make this a stronger, more honestly-validated claim.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split

# --- BEFORE: naive random split (no grouping, ignores that rows share clients) ---
train_naive, test_naive = train_test_split(merged_model, test_size=0.2, random_state=42)

overlap_naive = set(train_naive['client_hash_id']) & set(test_naive['client_hash_id'])
print(f"[BEFORE - random split] Train: {len(train_naive)} rows | Test: {len(test_naive)} rows")
print(f"[BEFORE - random split] Client overlap between train/test: {len(overlap_naive)} clients")

X_tr_n, y_tr_n = train_naive[feature_cols], train_naive['label_still_underperforming']
X_te_n, y_te_n = test_naive[feature_cols], test_naive['label_still_underperforming']

model_naive = LogisticRegression(max_iter=1000, class_weight='balanced')
model_naive.fit(X_tr_n, y_tr_n)
test_naive = test_naive.copy()
test_naive['model_score'] = model_naive.predict_proba(X_te_n)[:, 1]

auc_naive = roc_auc_score(y_te_n, test_naive['model_score'])
print(f"[BEFORE - random split] Test AUC: {auc_naive:.3f}")

results_naive = []
for k in [20, 50, 100]:
    p = precision_at_k(test_naive['model_score'], test_naive['label_still_underperforming'], k)
    results_naive.append({'k': k, 'precision': p})
print(pd.DataFrame(results_naive))

# --- AFTER: your existing grouped + time-aware split from ML-08 ---
print("\n[AFTER - grouped split] Test AUC:", round(roc_auc_score(y_test, test_df['model_score']), 3))
print(comparison_table[['k', 'model_precision']])

print(f"\nGAP: random-split AUC {auc_naive:.3f} vs grouped-split AUC {roc_auc_score(y_test, test_df['model_score']):.3f}")

[BEFORE - random split] Train: 126232 rows | Test: 31558 rows
[BEFORE - random split] Client overlap between train/test: 44 clients
[BEFORE - random split] Test AUC: 0.876
     k  precision
0   20       0.70
1   50       0.70
2  100       0.74

[AFTER - grouped split] Test AUC: 0.832
     k  model_precision
0   20             0.95
1   50             0.80
2  100             0.78

GAP: random-split AUC 0.876 vs grouped-split AUC 0.832


**Before/after: does the split honesty matter here?** Under a naive random split, `client_hash_id` overlap between train and test was [paste overlap count] clients — the model could partly memorize per-client baselines rather than learn the general pattern. Under the grouped + time-aware split (April labels, zero client overlap), AUC was 0.832; under the naive random split, AUC was [paste number]. [If naive is meaningfully higher: "The gap of X points shows some of the random-split score was the model recognizing clients it had already seen, not general signal — the grouped number is the honest one." If the gap is small: "The gap here is small, which is itself informative — it suggests client identity isn't doing much of the work for this particular feature set, though the grouped split remains the correct design regardless."]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# 1. Grain check on merged_model (should return zero rows)
grain_check = merged_model.groupby(['client_hash_id', 'content_hash_id']).size().reset_index(name='n')
print("Grain violations (should be empty):")
print(grain_check[grain_check['n'] > 1].head())

# 2. Timeline check — are all features strictly from March, label strictly from April?
print("\nFeature columns (all March-derived):", feature_cols)
print("Label column (April-derived):", 'label_still_underperforming')
print("No overlap by construction — March aggregates predate April's report_date range.")

# 3. Product-flag check — confirm no FlyRank health_score / flag columns snuck into features
suspect_cols = [c for c in merged_model.columns if 'flag' in c.lower() or 'health' in c.lower() or 'score' in c.lower()]
print("\nColumns matching flag/health/score pattern:", suspect_cols)
print("baseline_score is excluded from feature_cols:", 'baseline_score' not in feature_cols)

# 4. Train-with/without test on the top coefficient (total_clicks) — sanity check it's not "too good"
feature_cols_no_clicks = [c for c in feature_cols if c != 'total_clicks']
X_train_nc, X_test_nc = train_df[feature_cols_no_clicks], test_df[feature_cols_no_clicks]

model_no_clicks = LogisticRegression(max_iter=1000, class_weight='balanced')
model_no_clicks.fit(X_train_nc, y_train)
auc_no_clicks = roc_auc_score(y_test, model_no_clicks.predict_proba(X_test_nc)[:, 1])

print(f"\nAUC with total_clicks:    {roc_auc_score(y_test, test_df['model_score']):.3f}")
print(f"AUC without total_clicks: {auc_no_clicks:.3f}")
print("A collapse toward ~0.5 would signal leakage; a small drop is expected and healthy.")

# 5. IDs used only for grouping/joining, never as features
print("\nclient_hash_id / content_hash_id in feature_cols:",
      any(c in feature_cols for c in ['client_hash_id', 'content_hash_id']))

Grain violations (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, n]
Index: []

Feature columns (all March-derived): ['total_impressions', 'total_clicks', 'avg_position', 'ctr', 'ctr_gap', 'ctr_ratio', 'days_with_data']
Label column (April-derived): label_still_underperforming
No overlap by construction — March aggregates predate April's report_date range.

Columns matching flag/health/score pattern: ['baseline_score']
baseline_score is excluded from feature_cols: True

AUC with total_clicks:    0.832
AUC without total_clicks: 0.777
A collapse toward ~0.5 would signal leakage; a small drop is expected and healthy.

client_hash_id / content_hash_id in feature_cols: False


**Leakage audit.** Grain check on `merged_model` returned zero violations — one row per client/content pair, as expected. All features (`total_impressions`, `total_clicks`, `avg_position`, `ctr`, `ctr_gap`, `ctr_ratio`, `days_with_data`) are aggregated from March; the label is defined purely from April's own GSC data — no window overlap, no feature computed from the label. No FlyRank product flags, health scores, or the `baseline_score` column are in `feature_cols` — the baseline rule is used only for comparison, never as an input. `client_hash_id` and `content_hash_id` are used solely for grouping and joining, never as model features. Removing `total_clicks` (the top coefficient) dropped AUC from 0.832 to [paste number] — a modest, expected drop rather than a collapse toward 0.5, which is what real label-leakage would look like. No leakage found in this feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from ML-08, Section 4):** "`total_clicks` has the largest coefficient by a clear margin (-0.176), meaning more clicks in March strongly *lowers* predicted probability of April underperformance."

**Rewritten, honest version:** In this dataset, higher `total_clicks` in March is associated with a lower predicted probability of April underperformance — the largest single association among the seven features tested (standardized coefficient -0.176). This is an observed association from one client portfolio and one month-pair; it does not establish that increasing clicks would cause improved April outcomes, since no intervention was tested and the two months share the same underlying content and clients. The finding supports using `total_clicks` as a useful ranking signal, not a claim about what drives the underlying behavior.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.